In [0]:
from pyspark.sql.functions import col, current_timestamp

# Read Bronze
bronze_df = spark.table("workspace.default.bronze_transactions")

# 1. Identify invalid records
invalid_df = bronze_df.filter(
    (col("Amount") < 0) |
    (~col("Class").isin(0, 1))
)

# 2. Save invalid records to quarantine
(
    invalid_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.quarantine_transactions")
)

# 3. Keep valid records
silver_df = bronze_df.filter(
    (col("Amount") >= 0) &
    (col("Class").isin(0, 1))
)

# 4. Remove exact duplicate records
silver_df = silver_df.dropDuplicates()

# 5. Add Silver processing timestamp
silver_df = silver_df.withColumn(
    "_silver_processed_time",
    current_timestamp()
)

# 6. Write Silver table
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.silver_transactions")
)

print("Silver table created successfully!")

Silver table created successfully!


In [0]:
print(
    "Bronze:",
    spark.table("workspace.default.bronze_transactions").count()
)

print(
    "Silver:",
    spark.table("workspace.default.silver_transactions").count()
)

print(
    "Quarantine:",
    spark.table("workspace.default.quarantine_transactions").count()
)

Bronze: 284807
Silver: 283726
Quarantine: 0
